# 04 — Control Variables: Beta, Momentum, Market Cap, ROE, DE Ratio

In [2]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import statsmodels.api as sm
import yfinance as yf
import yf_cache
from pathlib import Path
import os

BASE = Path(os.getcwd())

QUARTER_ENDS = {
    'Q4FY21': '2021-03-31',
    'Q1FY22': '2021-06-30', 'Q2FY22': '2021-09-30',
    'Q3FY22': '2021-12-31', 'Q4FY22': '2022-03-31',
    'Q1FY23': '2022-06-30', 'Q2FY23': '2022-09-30',
    'Q3FY23': '2022-12-31', 'Q4FY23': '2023-03-31',
    'Q1FY24': '2023-06-30', 'Q2FY24': '2023-09-30',
    'Q3FY24': '2023-12-31', 'Q4FY24': '2024-03-31',
    'Q1FY25': '2024-06-30', 'Q2FY25': '2024-09-30',
    'Q3FY25': '2024-12-31', 'Q4FY25': '2025-03-31',
    'Q1FY26': '2025-06-30', 'Q2FY26': '2025-09-30',
    'Q3FY26': '2025-12-31', 'Q4FY26': '2026-03-31',
}
FY_ENDS = {
    'FY23': '2023-03-31', 'FY24': '2024-03-31',
    'FY25': '2025-03-31', 'FY26': '2026-03-31',
}
Q_TO_FY = {q: q[2:] for q in QUARTER_ENDS}

im = pd.read_excel(BASE / 'data' / 'processed' / 'industry_map.xlsx')
im = im.dropna(subset=['NSE Symbol']).copy()
im['BSE Code'] = pd.to_numeric(im['BSE Code'], errors='coerce')
tickers_ns = [f"{s}.NS" for s in im['NSE Symbol']]
print(f"Companies : {len(tickers_ns)}")
print(f"Quarters  : {len(QUARTER_ENDS)}")


Companies : 247
Quarters  : 21


In [3]:
print("Downloading NIFTY 50 ...")
nifty = yf_cache.download('^NSEI', start='2021-01-01', end='2026-04-01',
                    auto_adjust=True, progress=False)['Close'].squeeze().pct_change().dropna()

print("Downloading stock prices...")
prices_raw = yf_cache.download(tickers_ns, start='2021-01-01', end='2026-04-01',
                         auto_adjust=True, progress=True)['Close']
valid = prices_raw.columns[prices_raw.isna().mean() < 0.5]
prices = prices_raw[valid]
rets   = prices.pct_change().dropna(how='all')

print(f"\nValid tickers : {len(valid)} / {len(tickers_ns)}")
print(f"Date range    : {rets.index[0].date()} → {rets.index[-1].date()}")


[********************  41%                       ]  101 of 247 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SUVENPHARMA.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PEL.NS"}}}
[********************  42%                       ]  103 of 247 completed$SUVENPHARMA.NS: possibly delisted; no timezone found
[********************* 44%                       ]  108 of 247 completed$PEL.NS: possibly delisted; no timezone found
[**********************51%                       ]  125 of 247 completed$ISEC.NS: possibly delisted; no timezone found
[**********************74%***********            ]  183 of 247 completed$GET&D.NS: possibly delisted; no timezone found
[**********************77%************           ]  190 of 247 completed$ZOMATO.NS: possibly delisted; no timezone found
[*********************100%***********************]  247 of


Valid tickers : 237 / 247
Date range    : 2021-01-04 → 2026-03-30


In [ ]:
log_rets  = np.log1p(rets)
log_nifty = np.log1p(nifty)
all_dates = rets.index

# Pre-build BSE Code lookup once
bse_lookup = im.set_index('NSE Symbol')['BSE Code'].to_dict()

records = []
for q_fy, q_end_str in QUARTER_ENDS.items():
    q_end = pd.Timestamp(q_end_str)
    end_idx = all_dates.searchsorted(q_end, side='right') - 1
    if end_idx < 0: continue

    # Beta window: trailing 252 trading days
    beta_start = max(0, end_idx - 251)
    beta_dates = all_dates[beta_start: end_idx + 1]

    # Momentum window: 252 days ending ~21 days before quarter-end (skip 1 month)
    mom_end_idx   = max(0, end_idx - 21)
    mom_start_idx = max(0, mom_end_idx - 251)
    mom_dates     = all_dates[mom_start_idx: mom_end_idx + 1]

    # Align nifty to stock trading dates (reindex, ffill, dropna)
    mkt_beta    = log_nifty.reindex(beta_dates).ffill().dropna()
    mkt_beta_ex = mkt_beta - (0.065 / 252)

    for ticker in valid:
        sym = ticker.replace('.NS', '')
        stk_beta = log_rets.loc[beta_dates, ticker]
        aligned  = pd.DataFrame({'y': stk_beta - (0.065/252), 'x': mkt_beta_ex}).dropna()

        beta_val = np.nan
        if len(aligned) >= 60:
            X = sm.add_constant(aligned['x'])
            try:
                beta_val = sm.OLS(aligned['y'], X).fit().params['x']
            except Exception:
                pass

        stk_mom = log_rets.loc[mom_dates, ticker].dropna()
        mom_val = np.expm1(stk_mom.sum()) if len(stk_mom) >= 21 else np.nan

        records.append({
            'NSE Symbol':  sym,
            'BSE Code':    bse_lookup.get(sym),
            'Q_FY':        q_fy,
            'Beta_Market': beta_val,
            'Momentum':    mom_val,
        })

beta_mom = pd.DataFrame(records)
print(f"Beta/Momentum computed: {len(beta_mom):,} rows")
print(f"Beta null  : {beta_mom['Beta_Market'].isna().sum()}")
print(f"Mom  null  : {beta_mom['Momentum'].isna().sum()}")
beta_mom.head()

In [ ]:
mktcap_records = []
shares_cache = {}

print("Fetching shares outstanding...")
for ticker in valid:
    sym = ticker.replace('.NS', '')
    try:
        info = yf_cache.Ticker(ticker).info
        shares_cache[sym] = info.get('sharesOutstanding', np.nan)
    except Exception:
        shares_cache[sym] = np.nan

for q_fy, q_end_str in QUARTER_ENDS.items():
    q_end = pd.Timestamp(q_end_str)
    for ticker in valid:
        sym = ticker.replace('.NS', '')
        price_slice = prices.loc[:q_end_str, ticker].dropna()
        price = float(price_slice.iloc[-1]) if not price_slice.empty else np.nan
        shares = shares_cache.get(sym, np.nan)
        mktcap = price * shares if (not np.isnan(price) and not np.isnan(shares)) else np.nan
        mktcap_records.append({
            'NSE Symbol': sym,
            'Q_FY':       q_fy,
            'Log_MarketCap': np.log(mktcap) if (mktcap and mktcap > 0) else np.nan,
        })

mktcap_df = pd.DataFrame(mktcap_records)
print(f"Log_MarketCap null: {mktcap_df['Log_MarketCap'].isna().sum()} / {len(mktcap_df)}")
mktcap_df.head()

In [ ]:
fin_records = []
print("Fetching annual financials...")
for ticker in valid:
    sym = ticker.replace('.NS','')
    try:
        t   = yf_cache.Ticker(ticker)
        inc = t.income_stmt
        bal = t.balance_sheet
        if inc.empty or bal.empty: continue
        inc.columns = pd.to_datetime(inc.columns)
        bal.columns = pd.to_datetime(bal.columns)

        for fy, fy_end_str in FY_ENDS.items():
            fy_date = pd.Timestamp(fy_end_str)
            def nearest(df):
                diffs = abs(df.columns - fy_date)
                idx = diffs.argmin()
                return df.columns[idx] if diffs[idx].days <= 180 else None

            ic, bc = nearest(inc), nearest(bal)
            rec = {'NSE Symbol': sym, 'FY': fy}
            ni  = float(inc.loc['Net Income', ic]) if (ic is not None and 'Net Income' in inc.index) else np.nan
            eq  = float(bal.loc['Stockholders Equity', bc]) if (bc is not None and 'Stockholders Equity' in bal.index) else np.nan
            debt = float(bal.loc['Total Debt', bc]) if (bc is not None and 'Total Debt' in bal.index) else np.nan
            rec['ROE']      = ni / eq   if (not np.isnan(ni) and not np.isnan(eq) and eq != 0) else np.nan
            rec['DE_Ratio'] = debt / eq if (not np.isnan(debt) and not np.isnan(eq) and eq != 0) else np.nan
            fin_records.append(rec)
    except Exception:
        pass

fin_df = pd.DataFrame(fin_records)
print(f"Financials: {len(fin_df):,} rows | ROE null: {fin_df['ROE'].isna().sum()} | DE null: {fin_df['DE_Ratio'].isna().sum()}")
fin_df.head()

In [7]:
# Merge beta/momentum + market cap
controls_q = beta_mom.merge(mktcap_df[['NSE Symbol','Q_FY','Log_MarketCap']],
                             on=['NSE Symbol','Q_FY'], how='left')

# Map annual financials to quarters via FY
controls_q['FY'] = controls_q['Q_FY'].map(Q_TO_FY)
controls_q = controls_q.merge(fin_df, on=['NSE Symbol','FY'], how='left')

# Forward-fill ROE/DE within firm (use last available FY value)
controls_q = controls_q.sort_values(['NSE Symbol','Q_FY'])
for col in ['ROE','DE_Ratio']:
    controls_q[col] = controls_q.groupby('NSE Symbol')[col].ffill()

CTRL_COLS = ['Beta_Market','Momentum','Log_MarketCap','ROE','DE_Ratio']
print("Final controls shape:", controls_q.shape)
print("\nNull counts:")
print(controls_q[CTRL_COLS].isnull().sum())
print(f"\nCompanies: {controls_q['NSE Symbol'].nunique()}")
print(f"Quarters : {controls_q['Q_FY'].nunique()}")

out = BASE / 'data' / 'processed' / 'controls_quarterly.csv'
controls_q.to_csv(out, index=False)
controls_q.head()

Final controls shape: (4977, 9)

Null counts:
Beta_Market      118
Momentum         116
Log_MarketCap     95
ROE              238
DE_Ratio         339
dtype: int64

Companies: 237
Quarters : 21


,NSE Symbol,BSE Code,Q_FY,Beta_Market,Momentum,Log_MarketCap,FY,ROE,DE_Ratio
237,360ONE,542772,Q1FY22,0.440197,0.182202,25.370093,FY22,NaN,NaN
1185,360ONE,542772,Q1FY23,0.549040,0.420095,25.658246,FY23,0.210743,2.172117
2133,360ONE,542772,Q1FY24,0.451880,0.201126,25.901960,FY24,0.233123,2.710693
3081,360ONE,542772,Q1FY25,0.737458,1.043030,26.696203,FY25,0.143714,1.560649
4029,360ONE,542772,Q1FY26,1.007149,0.221440,26.902614,FY26,0.143714,1.560649
